## 1. Imports & Configuration

In [1]:
import os, json, pickle, time, warnings, re, requests
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn
from transformers import BertTokenizer, BertModel

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

PROJECT_ROOT    = Path(os.getcwd()).parent
DATA_DIR        = PROJECT_ROOT / 'data'
RESULTS_DIR     = PROJECT_ROOT / 'results'
MODELS_DIR      = PROJECT_ROOT / 'models'

STARPEP_CSV     = DATA_DIR / 'StarPep_APD_only.csv'
PREDICTIONS_CSV = RESULTS_DIR / 'test_predictions.csv'
# L2-normalised embeddings — cosine similarity = dot product of unit vectors
EMBEDDINGS_NPY  = DATA_DIR / 'starpep_embeddings_cosine.npy'
METADATA_PKL    = DATA_DIR / 'starpep_metadata.pkl'
OUTPUT_CSV      = RESULTS_DIR / 'amp_rag_results.csv'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

PROTBERT_ID = 'Rostlab/prot_bert_bfd'
MAX_LEN     = 128
EMBED_BATCH = 16
TOP_K       = 3

OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_MODEL    = 'biomistral:7b'
LLM_TEMPERATURE = 0
LLM_MAX_TOKENS  = 800

if torch.cuda.is_available():           DEVICE = 'cuda'
elif torch.backends.mps.is_available(): DEVICE = 'mps'
else:                                   DEVICE = 'cpu'

def clear_memory():
    if DEVICE == 'mps':  torch.mps.empty_cache()
    elif DEVICE == 'cuda': torch.cuda.empty_cache()

print(f'Device     : {DEVICE}')
print(f'Output CSV : {OUTPUT_CSV.name}')
print(f'LLM        : {OLLAMA_MODEL}  @ {OLLAMA_BASE_URL}')
print(f'Temperature: {LLM_TEMPERATURE}  (deterministic)')
print(f'Similarity : cosine (L2-normalised dot product, magnitude-invariant)')

## 1b. Ollama Health Check

Verifies that Ollama is running locally and `biomistral:7b` is available.  
Start Ollama with: `ollama serve` (or open the Ollama desktop app).

In [2]:
# ── Verify Ollama is reachable and model is loaded ─────────────────────────────
def ollama_health_check(base_url: str, model: str) -> bool:
    """Return True if Ollama is running and the model is available."""
    try:
        # Check server is up
        r = requests.get(f"{base_url}/api/tags", timeout=5)
        r.raise_for_status()
        models_available = [m["name"] for m in r.json().get("models", [])]
        print(f"Ollama server   : ✅ running")
        print(f"Models loaded   : {models_available}")

        if model in models_available:
            print(f"Target model    : ✅ {model} ready")
            return True
        else:
            print(f"Target model    : ❌ '{model}' not found")
            print(f"  Run: ollama pull {model}")
            return False

    except requests.exceptions.ConnectionError:
        print("❌ Ollama not reachable — start it with: ollama serve")
        return False
    except Exception as e:
        print(f"❌ Health check failed: {e}")
        return False

OLLAMA_READY = ollama_health_check(OLLAMA_BASE_URL, OLLAMA_MODEL)

if OLLAMA_READY:
    # Quick warm-up call to load model weights into memory
    print("\nWarm-up call (loads model weights)...")
    r = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model":    OLLAMA_MODEL,
            "messages": [{"role": "user", "content": "What is an antimicrobial peptide? One sentence."}],
            "stream":   False,
            "options":  {"temperature": LLM_TEMPERATURE, "num_predict": 60},
        },
        timeout=120,
    )
    warmup_reply = r.json()["message"]["content"].strip()
    print(f"✅ Model warm-up OK: {warmup_reply[:120]}")
else:
    print("\n⚠️  Fix Ollama before running Section 6.")

In [3]:
import subprocess, requests as _req
from pathlib import Path
from tqdm.auto import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────
MODEL_DIR    = PROJECT_ROOT / 'models' / 'biomistral'
GGUF_PATH    = MODEL_DIR / 'ggml-model-Q4_K_M.gguf'
MODELFILE_PATH = MODEL_DIR / 'Modelfile'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Download GGUF (skip if already present) ─────────────────────────────────
GGUF_URL = (
    'https://huggingface.co/BioMistral/BioMistral-7B-GGUF/resolve/main/'
    'ggml-model-Q4_K_M.gguf'
)
if GGUF_PATH.exists():
    print(f'GGUF already present: {GGUF_PATH} ({GGUF_PATH.stat().st_size / 1e9:.2f} GB)')
else:
    print('Downloading BioMistral-7B Q4_K_M GGUF (~4.1 GB) …')
    with _req.get(GGUF_URL, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(GGUF_PATH, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
    print(f'Downloaded: {GGUF_PATH}')

# ── 2. Write Ollama Modelfile ──────────────────────────────────────────────────
MODELFILE_CONTENT = f"""FROM {GGUF_PATH}
TEMPLATE \"\"\"<s>[INST] {{{{ if .System }}}}{{{{ .System }}}}
{{{{ end }}}}{{{{ .Prompt }}}} [/INST]\"\"\"
SYSTEM \"\"\"You are BioMistral, a biomedical AI assistant trained on PubMed Central.
Provide detailed, accurate, and scientifically rigorous biomedical explanations.\"\"\"
PARAMETER stop "[INST]"
PARAMETER stop "[/INST]"
PARAMETER stop "</s>"
PARAMETER temperature 0
PARAMETER num_predict 800
"""
MODELFILE_PATH.write_text(MODELFILE_CONTENT)
print('Modelfile written.')

# ── 3. Register with Ollama ────────────────────────────────────────────────────
print('Running: ollama create biomistral:7b …')
result = subprocess.run(
    ['ollama', 'create', 'biomistral:7b', '-f', str(MODELFILE_PATH)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('ollama create failed')

# ── 4. Verify ──────────────────────────────────────────────────────────────────
resp = _req.get(f'{OLLAMA_BASE_URL}/api/tags')
names = [m['name'] for m in resp.json().get('models', [])]
if any('biomistral' in n for n in names):
    print('✅ biomistral:7b is registered and ready.')
else:
    print('⚠️  biomistral:7b not found. Models available:', names)

## 2. Load StarPep Database

In [4]:
# ── Load StarPep ────────────────────────────────────────────────────────────────
starpep_df = pd.read_csv(STARPEP_CSV)

# Clean up sequences — remove any non-standard characters
starpep_df["sequence"] = starpep_df["sequence"].astype(str).str.strip().str.upper()
starpep_df = starpep_df[starpep_df["sequence"].str.len() > 0].reset_index(drop=True)

# Fill NA in metadata columns for safe downstream use
meta_cols = ["assessedAgainst", "relatedTo", "producedBy", "modifiedBy",
             "compiledIn", "linkedTo", "constitutedBy"]
for col in meta_cols:
    if col in starpep_df.columns:
        starpep_df[col] = starpep_df[col].fillna("").astype(str)

print(f"StarPep records     : {len(starpep_df):,}")
print(f"Columns             : {starpep_df.columns.tolist()}")
print(f"\nSequence length stats:")
print(starpep_df["length"].describe().round(1))
starpep_df.head(3)

## 3. Build / Load Vector Store

In [5]:
# ── Load ProtBERT tokenizer and encoder ────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained(
    PROTBERT_ID, do_lower_case=False, cache_dir=str(MODELS_DIR)
)
encoder = BertModel.from_pretrained(PROTBERT_ID, cache_dir=str(MODELS_DIR))
encoder = encoder.to(DEVICE)
encoder.eval()

total_params = sum(p.numel() for p in encoder.parameters())
print(f"✅ ProtBERT-BFD loaded  ({total_params/1e6:.0f}M params, device={DEVICE})")

In [6]:
@torch.no_grad()
def embed_sequences(sequences: list[str],
                    tokenizer,
                    model,
                    device: str,
                    batch_size: int = EMBED_BATCH,
                    max_len: int = MAX_LEN) -> np.ndarray:
    """
    Embed a list of amino-acid sequences using ProtBERT-BFD.
    Returns a (N, 1024) float32 numpy array of [CLS] embeddings.
    """
    all_embeddings = []
    n_batches = (len(sequences) + batch_size - 1) // batch_size

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i : i + batch_size]
        batch_num = i // batch_size + 1

        # ProtBERT requires space-separated amino acids
        spaced = [" ".join(list(seq.upper().replace(" ", ""))) for seq in batch]

        enc = tokenizer(
            spaced,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(device)
        attn_mask = enc["attention_mask"].to(device)

        out = model(input_ids=input_ids, attention_mask=attn_mask)
        cls_emb = out.last_hidden_state[:, 0, :].cpu().float().numpy()
        all_embeddings.append(cls_emb)

        del input_ids, attn_mask, out
        clear_memory()

        if batch_num % 20 == 0 or batch_num == n_batches:
            pct = batch_num / n_batches * 100
            print(f"  [{batch_num:4d}/{n_batches}]  {pct:5.1f}%  "
                  f"processed {min(i+batch_size, len(sequences)):4d}/{len(sequences)} seqs",
                  flush=True)

    return np.vstack(all_embeddings)   # (N, 1024)


print("Embedding function defined.")

In [7]:
if EMBEDDINGS_NPY.exists() and METADATA_PKL.exists():
    print('Cached embeddings found — loading...')
    store_embeddings = np.load(EMBEDDINGS_NPY)
    with open(METADATA_PKL, 'rb') as f:
        store_metadata = pickle.load(f)
    print(f'   Shape : {store_embeddings.shape}  (L2-normalised unit vectors)')
    print(f'   Meta  : {len(store_metadata)} records')
else:
    print(f'Embedding {len(starpep_df):,} StarPep sequences...')
    t0 = time.time()
    raw_embeddings = embed_sequences(starpep_df['sequence'].tolist(),
                                     tokenizer, encoder, DEVICE)
    # L2-normalise: each row becomes a unit vector
    # After this: A @ B.T = cosine_similarity(A, B)  for all row pairs
    norms = np.linalg.norm(raw_embeddings, axis=1, keepdims=True)
    store_embeddings = raw_embeddings / np.clip(norms, 1e-9, None)
    np.save(EMBEDDINGS_NPY, store_embeddings)
    print(f'\nL2-normalised embeddings saved -> {EMBEDDINGS_NPY.name}  {store_embeddings.shape}')

    store_metadata = []
    for _, row in starpep_df.iterrows():
        store_metadata.append({
            'id':              row['id'],
            'sequence':        row['sequence'],
            'length':          row.get('length', len(row['sequence'])),
            'assessedAgainst': row.get('assessedAgainst', ''),
            'relatedTo':       row.get('relatedTo', ''),
            'producedBy':      row.get('producedBy', ''),
            'modifiedBy':      row.get('modifiedBy', ''),
            'compiledIn':      row.get('compiledIn', ''),
            'linkedTo':        row.get('linkedTo', ''),
        })
    with open(METADATA_PKL, 'wb') as f:
        pickle.dump(store_metadata, f)
    print(f'Metadata saved -> {METADATA_PKL.name}  ({len(store_metadata)} records)')
    print(f'Time : {(time.time()-t0)/60:.1f} min')

# Verify unit norms (should all be ~1.0)
sample_norms = np.linalg.norm(store_embeddings[:5], axis=1)
print(f'\nSample L2 norms (should be 1.0): {sample_norms.round(6)}')
print(f'Vector store ready — {store_embeddings.shape[0]:,} unit-norm AMPs @ {store_embeddings.shape[1]}-d')
print('Similarity: cosine  (store @ query.T  since both are unit vectors)')

## 4. Load Positive AMP Predictions

Load the test set predictions from `results/test_predictions.csv` and filter to **positive AMP predictions** (`predicted == 1`).  
These are the sequences we will query against the StarPep vector store.

In [8]:
pred_df = pd.read_csv(PREDICTIONS_CSV)
print(f"Total test predictions : {len(pred_df):,}")
print(f"Class distribution     : {pred_df['predicted'].value_counts().to_dict()}")

# ── Filter to positive (predicted AMP) ─────────────────────────────────────────
pos_df = pred_df[pred_df["predicted"] == 1].reset_index(drop=True)
print(f"\nPositive AMP predictions : {len(pos_df):,}")
print(f"  True Positives (TP)    : {(pos_df['result']=='TP').sum()}")
print(f"  False Positives (FP)   : {(pos_df['result']=='FP').sum()}")
pos_df.head(5)

## 5. Cosine Similarity Retrieval — Top-3

In [9]:
def retrieve_top_k(query_seq, store_embs, store_meta, tokenizer, model, device, top_k=TOP_K):
    # 1. Embed query
    query_emb = embed_sequences([query_seq], tokenizer, model, device, batch_size=1)
    # 2. L2-normalise to unit vector
    query_unit = query_emb / np.clip(np.linalg.norm(query_emb), 1e-9, None)
    # 3. Cosine similarity = dot product of unit vectors
    #    store_embs already L2-normalised; scores in [-1, 1]
    cosine_scores = (store_embs @ query_unit.T).squeeze()
    # 4. Top-k descending — NO filtering, self-match included intentionally
    top_idx = np.argsort(cosine_scores)[::-1][:top_k]
    results = []
    for rank, idx in enumerate(top_idx, start=1):
        meta = dict(store_meta[idx])
        meta['rank']        = rank
        meta['cosine_sim']  = round(float(cosine_scores[idx]), 6)
        # Flag exact self-match (score=1.0 means query exists in StarPep)
        meta['is_self_match'] = (meta['cosine_sim'] >= 0.9999 and
                                  store_meta[idx]['sequence'].upper() == query_seq.upper())
        results.append(meta)
    return results

print('Retrieval function ready — cosine similarity, self-match included (score=1.0 flags known AMP).')

In [10]:
demo_seq = pos_df['sequence'].iloc[0]
print(f"Query : {demo_seq}  (len={len(demo_seq)} aa,  prob={pos_df['prob_amp'].iloc[0]:.4f})")
print('\nRetrieving top-3 via cosine similarity (self-match shown if present)...')
demo_results = retrieve_top_k(demo_seq, store_embeddings, store_metadata,
                               tokenizer, encoder, DEVICE)
print(f"\n{'Rank':<5} {'CosineSim':>10}  {'Self?':>5}  {'ID':<18} {'Seq':<30} {'Activities (first 3)'}")
print('-' * 115)
for r in demo_results:
    acts = ', '.join(r['relatedTo'].split('|')[:3]) if r['relatedTo'] else 'N/A'
    flag = ' ✓ SELF' if r['is_self_match'] else ''
    print(f"#{r['rank']:<4} {r['cosine_sim']:>10.6f}  {'YES' if r['is_self_match'] else 'no':>5}  {r['id']:<18} {r['sequence'][:28]:<30} {acts}{flag}")
print()
self_match = next((r for r in demo_results if r['is_self_match']), None)
if self_match:
    print(f"✅ Exact self-match found at rank {self_match['rank']} — this sequence IS in StarPep APD (cosine=1.0)")
else:
    print('ℹ️  No exact self-match in top-3 — sequence is not in StarPep APD (or only partially similar)')

### Run retrieval for all positive AMPs

In [11]:
print(f"Retrieving top-{TOP_K} similar AMPs for {len(pos_df):,} positive predictions...")
print("(Each query embeds one sequence — may take a few minutes)\n")

retrieval_results = {}   # seq → list of top-k dicts

for idx, row in pos_df.iterrows():
    seq = row["sequence"]
    hits = retrieve_top_k(seq, store_embeddings, store_metadata,
                          tokenizer, encoder, DEVICE)
    retrieval_results[seq] = {
        "sequence":  seq,
        "prob_amp":  row["prob_amp"],
        "true_label":int(row["label"]),
        "result":    row["result"],
        "hits":      hits,
    }
    if (idx + 1) % 50 == 0 or (idx + 1) == len(pos_df):
        print(f"  [{idx+1:>3}/{len(pos_df)}] done", flush=True)

clear_memory()
print(f"\n✅ Retrieval complete for {len(retrieval_results):,} sequences.")

## 6. LLM Biological Explanation

In [12]:
# ── Physicochemical property helpers ───────────────────────────────────────────
HYDROPHOBIC = set("AVILMFYW")
CHARGED_POS = set("KRH")
CHARGED_NEG = set("DE")

def compute_properties(seq: str) -> dict:
    seq = seq.upper()
    n = len(seq)
    hydrophobic_count = sum(1 for aa in seq if aa in HYDROPHOBIC)
    pos_charge  = sum(1 for aa in seq if aa in CHARGED_POS)
    neg_charge  = sum(1 for aa in seq if aa in CHARGED_NEG)
    net_charge  = pos_charge - neg_charge
    return {
        "length":          n,
        "hydrophobicity":  round(hydrophobic_count / n * 100, 1),
        "net_charge":      net_charge,
        "pos_charged_pct": round(pos_charge / n * 100, 1),
        "neg_charged_pct": round(neg_charge / n * 100, 1),
    }


def format_activities(pipe_str: str, max_items: int = 6) -> str:
    """Convert pipe-separated string to a comma-separated short list."""
    items = [x.strip() for x in pipe_str.split("|") if x.strip()][:max_items]
    return ", ".join(items) if items else "Not specified"


def format_targets(pipe_str: str, max_items: int = 8) -> str:
    items = [x.strip() for x in pipe_str.split("|") if x.strip()][:max_items]
    return ", ".join(items) if items else "Not specified"


def build_prompt(query_seq: str,
                 prob_amp: float,
                 hits: list[dict]) -> str:
    """
    Construct an expert-level biological prompt for the LLM.
    """
    props = compute_properties(query_seq)

    # Format the three retrieved AMPs
    hit_sections = []
    for h in hits:
        activities = format_activities(h.get("relatedTo", ""))
        targets    = format_targets(h.get("assessedAgainst", ""))
        source     = format_activities(h.get("producedBy", ""), max_items=3)
        mods       = format_activities(h.get("modifiedBy", ""), max_items=3)

        hit_sections.append(
            f"  [{h['rank']}] ID: {h['id']}  Similarity: {h['cosine_sim']:.4f}\n"
            f"      Sequence     : {h['sequence']}  (length: {h.get('length', len(h['sequence']))} aa)\n"
            f"      Activities   : {activities}\n"
            f"      Target organisms: {targets}\n"
            f"      Biological source: {source}\n"
            f"      Modifications : {mods if mods != 'Not specified' else 'None reported'}"
        )

    hits_text = "\n\n".join(hit_sections)

    prompt = f"""You are an expert computational biologist specialising in antimicrobial peptides (AMPs) and protein bioinformatics.

A machine-learning classifier (ProtBERT-BFD fine-tuned on the Veltri dataset) has predicted the following peptide sequence to be an antimicrobial peptide with probability {prob_amp:.4f}:

QUERY SEQUENCE: {query_seq}

PHYSICOCHEMICAL PROPERTIES:
  • Length         : {props['length']} amino acids
  • Hydrophobicity : {props['hydrophobicity']}% hydrophobic residues ({', '.join(HYDROPHOBIC & set(query_seq.upper()))})
  • Net charge     : {props['net_charge']:+d} (Positive residues: {props['pos_charged_pct']}%, Negative: {props['neg_charged_pct']}%)

TOP-3 MOST SIMILAR KNOWN AMPs (cosine similarity in ProtBERT-BFD embedding space):

{hits_text}

Based on the query sequence and its closest known AMP neighbours, provide a comprehensive biological explanation covering:

1. ANTIMICROBIAL MECHANISM — What membrane-disrupting or intracellular mechanisms are most likely? Relate to physicochemical properties (charge, hydrophobicity, amphipathicity).
2. PREDICTED TARGET SPECTRUM — Which organisms or pathogens is this peptide most likely active against, based on the neighbours' target profiles?
3. STRUCTURAL FEATURES — What structural motifs (alpha-helix, beta-sheet, random coil) and key residues likely drive activity?
4. BIOLOGICAL ORIGIN & EVOLUTION — What biological context might this peptide arise from, based on the source organisms of similar peptides?
5. THERAPEUTIC POTENTIAL — What biomedical applications (antibacterial, antifungal, antiviral, anticancer) are most supported by the evidence?
6. RESEARCH RECOMMENDATIONS — What experimental assays would you recommend to validate this peptide's activity?

Be specific, cite amino acid positions where relevant, and ground your explanation in the retrieved AMP data."""

    return prompt


print("Prompt builder defined.")
print("\nSample prompt for demo sequence:")
print("-"*60)
sample_prompt = build_prompt(
    demo_seq, pos_df['prob_amp'].iloc[0], demo_results
)
print(sample_prompt[:600], "...")

In [13]:
print(sample_prompt)

In [14]:
# ── Ollama LLM inference (local, temperature=0 → deterministic) ────────────────
def query_llm(prompt: str,
              model:  str    = OLLAMA_MODEL,
              base_url: str  = OLLAMA_BASE_URL,
              max_tokens: int = LLM_MAX_TOKENS,
              temperature: float = LLM_TEMPERATURE,
              retries: int = 3) -> str:
    """
    Send a chat prompt to the local Ollama instance.
    temperature=0 → greedy decoding → fully deterministic output.
    Returns the generated explanation text.
    """
    payload = {
        "model":  model,
        "stream": False,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert computational biologist specialising in "
                    "antimicrobial peptides (AMPs). Provide detailed, scientifically "
                    "accurate biological explanations using structured headings. "
                    "Base your answer strictly on the provided sequence data and "
                    "retrieved similar AMPs."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        "options": {
            "temperature": temperature,   # 0 = deterministic
            "num_predict": max_tokens,
            "top_p":       1.0,           # disable nucleus sampling at temp=0
            "top_k":       1,             # greedy: always pick the highest-prob token
        },
    }

    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(
                f"{base_url}/api/chat",
                json=payload,
                timeout=300,   # allow up to 5 min for long generations
            )
            resp.raise_for_status()
            return resp.json()["message"]["content"].strip()
        except requests.exceptions.Timeout:
            print(f"  ⚠️  Timeout on attempt {attempt}/{retries}. Retrying...")
        except requests.exceptions.ConnectionError:
            print(f"  ❌ Ollama not reachable. Is 'ollama serve' running?")
            return "[LLM ERROR] Ollama not reachable."
        except Exception as e:
            wait = 2 ** attempt
            print(f"  ⚠️  Attempt {attempt}/{retries} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)

    return f"[LLM ERROR] Could not generate explanation after {retries} attempts."


print(f"✅ Ollama LLM function ready")
print(f"   Model       : {OLLAMA_MODEL}")
print(f"   Temperature : {LLM_TEMPERATURE}  (greedy / deterministic)")
print(f"   top_k       : 1  (greedy decoding)")
print(f"   Max tokens  : {LLM_MAX_TOKENS}")

### Generate explanations for all positive AMP predictions

In [15]:
if not OLLAMA_READY:
    raise RuntimeError("Ollama is not running. Start it with: ollama serve")

print(f"Generating biological explanations for {len(retrieval_results):,} sequences...")
print(f"LLM   : {OLLAMA_MODEL}  (local Ollama)")
print(f"Temp  : {LLM_TEMPERATURE}  — deterministic greedy decoding\n")

explanations = {}   # seq → explanation text
failed_seqs  = []
t_start = time.time()

for i, (seq, data) in enumerate(list(retrieval_results.items())[:3], start=1):
    t0          = time.time()
    prompt      = build_prompt(seq, data["prob_amp"], data["hits"])
    explanation = query_llm(prompt)
    elapsed_seq = time.time() - t0

    explanations[seq] = explanation

    status = "✅" if "[LLM ERROR]" not in explanation else "❌"
    if "[LLM ERROR]" in explanation:
        failed_seqs.append(seq)

    print(f"  {status} [{i:>3}/{len(retrieval_results)}]  "
          f"{elapsed_seq:5.1f}s  seq={seq[:18]:<18}  "
          f"words={len(explanation.split()):>4}", flush=True)

total_elapsed = time.time() - t_start
print(f"\n{'─'*60}")
print(f"✅ Explanations generated : {len(explanations) - len(failed_seqs):,}")
print(f"❌ Failed                : {len(failed_seqs)}")
print(f"⏱  Total time            : {total_elapsed/60:.1f} min  "
      f"({total_elapsed/len(retrieval_results):.1f}s per sequence)")

## 7. Save All Results to Single CSV

In [16]:
def fmt(pipe_str, max_items=8):
    items = [x.strip() for x in str(pipe_str).split('|') if x.strip()][:max_items]
    return ', '.join(items)

rows = []
for idx, (seq, data) in enumerate(retrieval_results.items()):
    explanation = explanations.get(seq, '[No explanation generated]')
    props = compute_properties(seq)
    row = {
        'seq_idx':           idx + 1,
        'query_sequence':    seq,
        'query_length':      props['length'],
        'prob_amp':          round(data['prob_amp'], 6),
        'true_label':        data['true_label'],
        'prediction_result': data['result'],
        'in_starpep':        any(h.get('is_self_match', False) for h in data['hits']),
        'hydrophobicity_pct':props['hydrophobicity'],
        'net_charge':        props['net_charge'],
        'pos_charge_pct':    props['pos_charged_pct'],
        'neg_charge_pct':    props['neg_charged_pct'],
    }
    for n in range(1, TOP_K + 1):
        p = f'hit{n}_'
        if n <= len(data['hits']):
            h = data['hits'][n - 1]
            row[p+'starpep_id']    = h['id']
            row[p+'is_self_match'] = h.get('is_self_match', False)
            row[p+'cosine_sim']   = round(h['cosine_sim'], 4)
            row[p+'sequence']      = h['sequence']
            row[p+'length']        = h.get('length', len(h['sequence']))
            row[p+'activities']    = fmt(h.get('relatedTo',       ''))
            row[p+'targets']       = fmt(h.get('assessedAgainst', ''), max_items=10)
            row[p+'source']        = fmt(h.get('producedBy',      ''), max_items=4)
            row[p+'modifications'] = fmt(h.get('modifiedBy',      ''), max_items=4)
        else:
            for col in ['starpep_id','cosine_sim','sequence','length',
                        'activities','targets','source','modifications']:
                row[p+col] = ''
    row['biological_explanation'] = explanation
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f'Results saved -> {OUTPUT_CSV}')
print(f'   Rows    : {len(results_df)}')
print(f'   Columns : {len(results_df.columns)}')
print(f'   Size    : {OUTPUT_CSV.stat().st_size / 1024:.1f} KB')
print()
print('Column layout:')
print('  [query]           seq_idx  query_sequence  query_length  prob_amp  true_label  prediction_result')
print('  [physicochemical] hydrophobicity_pct  net_charge  pos_charge_pct  neg_charge_pct')
for n in range(1, TOP_K + 1):
    print(f'  [hit {n}]           hit{n}_starpep_id  hit{n}_cosine_sim  hit{n}_sequence  hit{n}_length')
    print(f'                    hit{n}_activities  hit{n}_targets  hit{n}_source  hit{n}_modifications')
print('  [explanation]     biological_explanation')
print()
display(results_df[['query_sequence','prob_amp','prediction_result',
                     'hit1_starpep_id','hit1_cosine_sim','hit1_activities']].head(3))

### Preview — first row from CSV

In [17]:
row = results_df.iloc[0]
print('=' * 80)
print(f"  seq_idx            : {row['seq_idx']}")
print(f"  query_sequence     : {row['query_sequence']}")
print(f"  prob_amp           : {row['prob_amp']:.4f}")
print(f"  prediction_result  : {row['prediction_result']}")
print(f"  net_charge         : {row['net_charge']:+d}")
print(f"  hydrophobicity_pct : {row['hydrophobicity_pct']}%")
print()
for n in range(1, TOP_K + 1):
    p = f'hit{n}_'
    print(f"  Hit #{n}: {row[p+'starpep_id']}  cosine_sim={row[p+'cosine_sim']:.2f}")
    print(f"    Sequence   : {row[p+'sequence']}")
    print(f"    Activities : {str(row[p+'activities'])[:80]}")
    print(f"    Targets    : {str(row[p+'targets'])[:80]}")
    print()
print('-' * 80)
print('  BIOLOGICAL EXPLANATION (Mistral-7B-Instruct, T=0):')
print('-' * 80)
print(row['biological_explanation'])
print('=' * 80)